In [8]:
from pathlib import Path
import subprocess
from broskill.processing.skill import SkillControl
from broskill.processing.tool import ToolControl
import sys

In [9]:
ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

In [10]:
sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)

In [11]:
sc.list_skills()

[Skill(name='create-prompt', description='Use when the user wants help writing or improving a prompt (for an LLM, agent, or skill) — asks clarifying questions about goal, audience, and constraints, then drafts the prompt and checks it against a quality checklist.', version='v0.1.0', path=WindowsPath('D:/broskill/skills/create-prompt'), tags=['create', 'prompt'], keywords=['create prompt', 'build prompt'], default=True, status='experiment'),
 Skill(name='create-skill', description="Use when the user wants to create a new Claude Code skill — asks clarifying questions about the skill's purpose and trigger conditions, then writes a SKILL.md with instructions and a checklist, progressively loading references/reference.md, references/script.md, or references/asset.md only when the new skill actually needs that kind of file.", version='v0.1.0', path=WindowsPath('D:/broskill/skills/create-skill'), tags=['create', 'skill'], keywords=['create skill', 'build skill'], default=False, status='experi

In [12]:
sc.load_skill('create-prompt')

"# Create Prompt\n\n## Step 1 — Clarify with the user\n\nAsk (skip anything already known):\n\n- What is this prompt for — what task, run against what model/agent, one-off or reusable?\n- Who/what is the audience (a model, a person, a downstream system)?\n- Required inputs, expected output format, and constraints (length, tone, structure)?\n- Any examples of good/bad output, or existing prompts to match the style of?\n\n## Step 2 — Draft\n\nWrite the prompt with:\n\n- A clear task statement up front\n- Only the context/constraints actually needed\n- An explicit output format, if one is expected\n- Examples only where they clarify something words alone can't\n\n## Step 3 — Checklist\n\n- [ ] Clarified purpose/audience before drafting\n- [ ] States the task unambiguously in the first lines\n- [ ] Includes only necessary context — no filler\n- [ ] Output format specified if relevant\n- [ ] Reviewed for ambiguity a reader/model could misinterpret\n\n## Step 4 — Confirm\n\nShow the draft ba

In [13]:
sc.load_skill_extension('create-skill', 'references/script.md')

'# Script practices\n\nConsult this file before writing any script into a skill\'s `scripts/` folder.\n\n## Required: `get_args()` for tool-schema loading\n\nEvery script must define a module-level `get_args()` that returns an `argparse.ArgumentParser` — `ToolControl.load_tool(skill_name, path)` (in `broskill.processing.tool`) imports the script and calls `get_args()` to auto-generate its `Tool`/`Arg` schema. Without it, `load_tool` raises `ValueError`.\n\n- `parser.description` becomes the tool\'s description — write it for whoever/whatever decides when to call the tool, not just as a docstring.\n- Each `add_argument` becomes an `Arg`: `dest` → name, `help` → description, `required` → required.\n- `type=` must be one of the types in `DTYPE_MAP` (`str`, `int`, `float`, `bool`) — anything else raises `ValueError` when the tool is loaded.\n- Don\'t reference `--flag` syntax in SKILL.md instructions; scripts are invoked as tools with named params (e.g. `path=...`), not raw CLI.\n\n```pyth

In [14]:
sc.load_skill_extension('read-file', 'scripts/read_file.py')

'import argparse\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).parent))\nfrom errors import InvalidPatternError, NoMatchError, MultipleMatchError, FileTooLargeError, UnreadableFileError\n\nMAX_READ_BYTES = 1_000_000\n\n\ndef get_args():\n    parser = argparse.ArgumentParser(\n        description="Read and print a single file\'s content. --path is resolved under the current directory with Path.rglob, so a bare filename or partial path works as long as it matches exactly one file."\n    )\n    parser.add_argument(\'--path\', type=str, required=True, help="filename or path/pattern to resolve with rglob")\n    return parser\n\n\ndef main():\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    args = get_args().parse_args()\n    pattern = args.path\n    root = Path.cwd()\n\n    if not pattern.strip():\n        raise InvalidPatternError(pattern, "pattern is empty")\n    if Path(pattern).is_absolute():\n        raise InvalidPatternError(pattern, "must be relat

In [15]:
tc.load_tool('read-file', 'scripts/read_file.py')

Tool(name='read_file', description="Read and print a single file's content. --path is resolved under the current directory with Path.rglob, so a bare filename or partial path works as long as it matches exactly one file.", args=[Arg(name='path', type='string', description='filename or path/pattern to resolve with rglob', required=True)], path=WindowsPath('D:/broskill/skills/read-file/scripts/read_file.py'))

In [17]:
tc.load_tool('read-file', 'scripts/list_files.py')

Tool(name='list_files', description="List files matching a glob pattern under the current directory, using Path.rglob (e.g. '*.md', 'skills/**/*.py').", args=[Arg(name='path', type='string', description="glob pattern for Path.rglob, e.g. '*.md'", required=True)], path=WindowsPath('D:/broskill/skills/read-file/scripts/list_files.py'))